# Dental Vision V1 — held-out validation v2
Disease-stratified 80/20 split, slower learning-rate decay, 50-epoch fine-tuning, and annotation sanity checks. The untouched 20% remains held out.


In [ ]:
!nvidia-smi
import os,shutil,pathlib,json,zipfile,subprocess,sys
%cd /kaggle/working
shutil.rmtree('/kaggle/working/dental-vision-v1',ignore_errors=True)
!git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1
%cd /kaggle/working/dental-vision-v1
!pip -q install -r requirements.txt


In [ ]:
root=pathlib.Path('data/dentex_validation'); shutil.rmtree(root,ignore_errors=True); root.mkdir(parents=True)
!python scripts/download_dentex.py --out data/dentex_validation --files training_data.zip
zpath=root/'training_data.zip'; wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted','impacted tooth','impacted teeth'}; candidates=[]
with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.lower().endswith('.json'): continue
        try: d0=json.loads(z.read(name))
        except Exception: continue
        if not isinstance(d0,dict) or not {'images','annotations'}.issubset(d0): continue
        cats=d0.get('categories_3') or d0.get('categories') or []; names={str(c.get('name','')).strip().lower() for c in cats}; score=len(names&wanted); bonus=2 if 'quadrant-enumeration-disease' in name.lower() else 0
        if score or bonus: candidates.append((score+bonus,len(d0['images']),name,d0,cats))
assert candidates
_,_,ann_member,d,cats=max(candidates,key=lambda x:(x[0],x[1])); d['categories']=cats
for a in d['annotations']:
    if 'category_id_3' in a: a['category_id']=a['category_id_3']
subset=root/'diagnostic'; (subset/'images').mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zpath) as z:
    members=z.namelist()
    for im in d['images']:
        fn=str(im['file_name']).replace('\\\\','/').lstrip('./'); matches=[m for m in members if m.endswith('/'+fn) or m==fn] or [m for m in members if pathlib.PurePosixPath(m).name==pathlib.PurePosixPath(fn).name]; src=matches[0]; dest=subset/'images'/pathlib.PurePosixPath(fn).name
        with z.open(src) as r,open(dest,'wb') as w: shutil.copyfileobj(r,w)
        im['file_name']=dest.name
(subset/'all.json').write_text(json.dumps(d)); zpath.unlink()
print('Prepared',len(d['images']),'labeled images')
\n# Fail fast on corrupt/misaligned annotations.\nfrom collections import Counter\ncat_ids={c['id'] for c in d['categories']}; ann_ids={a['category_id'] for a in d['annotations']}\nassert ann_ids <= cat_ids, (ann_ids,cat_ids)\nbad=[a for a in d['annotations'] if len(a.get('bbox',[]))!=4 or a['bbox'][2]<=0 or a['bbox'][3]<=0]\nassert not bad, f'Invalid boxes: {len(bad)}'\nprint('DISEASE CATEGORIES:',[(c['id'],c['name']) for c in d['categories']])\nprint('ANNOTATIONS/CLASS:',Counter(a['category_id'] for a in d['annotations']))\nprint('BBOX sanity passed:',len(d['annotations']),'boxes')\n

In [ ]:
!python scripts/split_dentex_holdout.py --annotations data/dentex_validation/diagnostic/all.json --train-out data/dentex_validation/diagnostic/train.json --val-out data/dentex_validation/diagnostic/val.json --val-fraction 0.20 --seed 20260920\n!python train.py --images data/dentex_validation/diagnostic/images --annotations data/dentex_validation/diagnostic/train.json --epochs 50 --batch-size 2 --lr 0.0025 --output /kaggle/working/dentex_holdout_v2.pt\n

In [ ]:
!python evaluate.py --images data/dentex_validation/diagnostic/images --annotations data/dentex_validation/diagnostic/val.json --checkpoint /kaggle/working/dentex_holdout_v2.pt --output /kaggle/working/dentex_validation_metrics_v2.json\nprint(pathlib.Path('/kaggle/working/dentex_validation_metrics_v2.json').read_text())\nprint('VALIDATION V2 COMPLETE: checkpoint + metrics are saved in Kaggle Output')\n